In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support
import shap
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Simulating the Saarthi dataset structure
N_SAMPLES = 10000
N_FEATURES = 2538

# 70:30 class imbalance (abandon vs complete)
# 0 = Abandon (70%), 1 = Complete (30%)
y = np.random.choice([0, 1], size=N_SAMPLES, p=[0.7, 0.3])

# Generating 2,538 features with ~28% missing values
X = np.random.randn(N_SAMPLES, N_FEATURES)
missing_mask = np.random.rand(*X.shape) < 0.28
X[missing_mask] = np.nan

X_df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(N_FEATURES)])
print(f"Dataset shape: {X_df.shape} | Target distribution: {np.bincount(y)}")

Dataset shape: (10000, 2538) | Target distribution: [6948 3052]


In [5]:
# Handle imbalance: ratio of negative to positive instances
scale_weight = (len(y) - sum(y)) / sum(y) 

# Base XGBoost Classifier parameters
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 6,
    'learning_rate': 0.05,
    'n_estimators': 300,
    'scale_pos_weight': scale_weight,
    'missing': np.nan, # XGBoost natively handles the 28% missing data
    'tree_method': 'hist', # Faster on Kaggle CPU/GPU
    'random_state': 42
}

base_clf = xgb.XGBClassifier(**xgb_params)

# Platt scaling for probability calibration as per deck spec
calibrated_clf = CalibratedClassifierCV(estimator=base_clf, method='sigmoid', cv=2)

# 5-fold Stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

roc_aucs = []
pr_aucs = []

print("Starting 5-Fold Cross Validation...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_df, y)):
    X_train, X_val = X_df.iloc[train_idx], X_df.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Train the calibrated classifier
    calibrated_clf.fit(X_train, y_train)
    
    # Predict probabilities
    y_pred_prob = calibrated_clf.predict_proba(X_val)[:, 1]
    
    # Evaluate
    fold_roc = roc_auc_score(y_val, y_pred_prob)
    fold_pr = average_precision_score(y_val, y_pred_prob)
    
    roc_aucs.append(fold_roc)
    pr_aucs.append(fold_pr)
    
    print(f"Fold {fold+1} | ROC-AUC: {fold_roc:.4f} | PR-AUC: {fold_pr:.4f}")

print(f"\nMean ROC-AUC: {np.mean(roc_aucs):.4f} (Target: 0.91)")
print(f"Mean PR-AUC: {np.mean(pr_aucs):.4f} (Target: 0.63)")

Starting 5-Fold Cross Validation...
Fold 1 | ROC-AUC: 0.4917 | PR-AUC: 0.3087
Fold 2 | ROC-AUC: 0.5066 | PR-AUC: 0.3061
Fold 3 | ROC-AUC: 0.4812 | PR-AUC: 0.3022
Fold 4 | ROC-AUC: 0.5103 | PR-AUC: 0.3135
Fold 5 | ROC-AUC: 0.4915 | PR-AUC: 0.2964

Mean ROC-AUC: 0.4963 (Target: 0.91)
Mean PR-AUC: 0.3054 (Target: 0.63)


In [7]:
# To use SHAP, we need a standard trained XGBoost model (without the calibrator wrapper)
# We will train one on the full dataset just for extracting SHAP values
final_model = xgb.XGBClassifier(**xgb_params)
final_model.fit(X_df, y)

# Initialize SHAP explainer
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_df)

# Function to get top 3 reason codes for a specific session (e.g., session index 0)
def get_reason_codes(session_idx, X_data, shap_vals, feature_names, top_n=3):
    session_shaps = shap_vals[session_idx]
    # Get indices of top absolute SHAP values
    top_indices = np.argsort(np.abs(session_shaps))[-top_n:][::-1]
    
    reasons = []
    for idx in top_indices:
        reasons.append({
            'feature': feature_names[idx],
            'value': X_data.iloc[session_idx, idx],
            'impact': 'Positive' if session_shaps[idx] > 0 else 'Negative',
            'shap_score': session_shaps[idx]
        })
    return reasons

# Example: Get reason codes for the first user session
top_reasons = get_reason_codes(0, X_df, shap_values, X_df.columns)
print("\nTop 3 Reason Codes for Session 0:")
for r in top_reasons:
    print(f"Feature: {r['feature']} | Impact: {r['impact']} | Score: {r['shap_score']:.4f}")


Top 3 Reason Codes for Session 0:
Feature: feature_1122 | Impact: Negative | Score: -0.3539
Feature: feature_801 | Impact: Negative | Score: -0.0784
Feature: feature_490 | Impact: Negative | Score: -0.0713


In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
import time
import json
import os

print("🔥 Initializing Saarthi Data Simulation...")
start_time = time.time()

# 1. Simulate the 10,000 Session Dataset
N_SAMPLES = 10000
N_FEATURES = 2538

# 70:30 class imbalance (0 = Abandon, 1 = Complete)
y = np.random.choice([0, 1], size=N_SAMPLES, p=[0.7, 0.3])

# Generate 2,538 features with ~28% missing values to mimic real banking telemetry
X = np.random.randn(N_SAMPLES, N_FEATURES)
missing_mask = np.random.rand(*X.shape) < 0.28
X[missing_mask] = np.nan

# Split for training and calibration
X_train, X_cal, y_train, y_cal = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"✅ Data generated. Shape: {X.shape}. Missing values injected.")
print("🚀 Training the XGBoost Base Model...")

# Handle class imbalance weight
scale_weight = (len(y_train) - sum(y_train)) / sum(y_train)

# Base XGBoost parameters for millisecond inference
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 6,
    'learning_rate': 0.05,
    'n_estimators': 300,
    'scale_pos_weight': scale_weight,
    'tree_method': 'hist',
    'random_state': 42
}

# 2. Train the Base Model
base_clf = xgb.XGBClassifier(**xgb_params)
base_clf.fit(X_train, y_train)

print("⚖️ Applying Platt Scaling (CalibratedClassifierCV)...")

# 3. Calibrate for accurate probability outputs
calibrated_clf = CalibratedClassifierCV(estimator=base_clf, method='sigmoid', cv='prefit')
calibrated_clf.fit(X_cal, y_cal)

# 4. Save the Model
# We extract the underlying XGBoost booster from the calibrated model to save it efficiently
model_path = os.path.join(os.path.dirname(__file__), '../backend_api/saarthi_xgb_model.json')
fitted_booster = calibrated_clf.calibrated_classifiers_[0].estimator.get_booster()
fitted_booster.save_model(model_path)

print(f"🎯 Boom. Model successfully trained in {time.time() - start_time:.2f} seconds.")
print(f"💾 Saved for production to: {model_path}")

🔥 Initializing Saarthi Data Simulation...
✅ Data generated. Shape: (10000, 2538). Missing values injected.
🚀 Training the XGBoost Base Model...
⚖️ Applying Platt Scaling (CalibratedClassifierCV)...


/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


NameError: name '__file__' is not defined

In [2]:
# ... (Keep everything above Step 3 exactly the same)

print("⚖️ Applying Platt Scaling...")

# 3. Calibrate for accurate probability outputs
calibrated_clf = CalibratedClassifierCV(estimator=base_clf, method='sigmoid', cv='prefit')
calibrated_clf.fit(X_cal, y_cal)

# 4. Save the Model (Kaggle-specific path)
# We point this directly to Kaggle's output folder so you can download it easily
model_path = '/kaggle/working/saarthi_xgb_model.json'
fitted_booster = calibrated_clf.calibrated_classifiers_[0].estimator.get_booster()
fitted_booster.save_model(model_path)

print(f"🎯 Boom. Model successfully trained.")
print(f"💾 Saved to Kaggle working directory: {model_path}")

⚖️ Applying Platt Scaling...
🎯 Boom. Model successfully trained.
💾 Saved to Kaggle working directory: /kaggle/working/saarthi_xgb_model.json


/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(
